# Stage 4 final-token probe accuracy by sequence-length quartile

Fit the original linear probes on the unchanged mixed binary/ternary training set, using only the final checkpoint (stage 4, step 2097152). Evaluate each fitted probe on four held-out length quartiles from one independently sampled **mixed** test pool of 65536 sequences (16384 per quartile).

Lengths count the complete sequence, including the final target token, and range from 8 to 27. Quartiles are equal-population groups, not equal-width length intervals. Discrete lengths can straddle a boundary: ties are shuffled with a fixed seed before sorting, so adjacent quartiles may share an endpoint length. No example index belongs to two groups. Sampling is independent of training; naturally recurring grammar sequences are not deduplicated, preserving the original distribution.

Training examples, seed, last-input-position representation, and probe hyperparameters stay unchanged. The same fitted probe at each layer is evaluated on all four groups. Separate quartile cache and figure filenames preserve other experiments. Run from the repository root. The final cell adds the existing saved stage 4 pure-ternary (length 27) curve and can regenerate the accuracy plot from the two saved result files alone.


## 1. Setup and probe datasets

In [ ]:
import gc, sys, pickle, warnings
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

WORKSPACE = Path.cwd()
if not (WORKSPACE / 'models').exists(): raise RuntimeError('Run from the repository root.')
sys.path[:0] = [str(WORKSPACE), str(WORKSPACE / 'datasets' / 'rhm')]
import models
from mixed_random_hierarchy_model import (
    sample_padded_rules, create_probabilities,
    sample_data_from_labels_varying_tree_tensorized,
)

V, L, BIN = 8, 3, 2
M_2, M_3, D_MAX, PAD_TOKEN = BIN, BIN * V, 3 ** L, V
T = D_MAX - 1
MODEL_DEPTH = 12
STEPS = [2097152]  # Stage 4 only.
SEED_RULES, TRAIN_SEED, TEST_SEED = 12345678, 42, 43
N_TRAIN, N_TEST = 32768, 65536  # Total test pool; 16384 per quartile.
QUARTILE_SEED = 44
QUARTILES = ('Q1', 'Q2', 'Q3', 'Q4')
NUM_HEADS, BATCH_SIZE, PROBE_C = 16, 256, 0.1
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
CHECKPOINTS = {step: WORKSPACE / f'run_L3_v8_bin2_ter16_P16777216_lr0.001_l20.03_bs128_depth12_demb_256_nheads_16_arr0_t{step}.pt' for step in STEPS}
RESULT_PATHS = {step: WORKSPACE / f'probe_results_step_{step}_mixed_train_length_quartiles_test_last_token_depth12.pkl' for step in STEPS}
RECOMPUTE = False
EXPERIMENT = 'mixed_train_length_quartiles_test_depth12_v1'
# Cached stages need no checkpoint file.
missing = [CHECKPOINTS[step] for step in STEPS
           if (RECOMPUTE or not RESULT_PATHS[step].is_file())
           and not CHECKPOINTS[step].is_file()]
if missing: raise FileNotFoundError('Missing checkpoints:\n' + '\n'.join(map(str, missing)))

rules = sample_padded_rules(V, V, M_2, M_3, 2, 3, L, SEED_RULES)
mixed_probs = create_probabilities(M_2, M_3, L)
def generate_sentences(n, probabilities, seed):
    state = torch.random.get_rng_state(); torch.manual_seed(seed)
    labels = torch.randint(0, V, (n,))
    sentences, _ = sample_data_from_labels_varying_tree_tensorized(labels, rules, probabilities, V, D_MAX)
    torch.random.set_rng_state(state)
    return sentences

def make_last_token_task(sentences):
    # Mirrors init.MySentenceDataLoader exactly.
    n = len(sentences); lengths = (sentences != PAD_TOKEN).sum(1)
    targets = sentences[torch.arange(n), lengths - 1]
    input_lengths = lengths - 1
    columns = torch.arange(T).unsqueeze(0).expand(n, T)
    source_columns = columns - T + input_lengths.unsqueeze(1)
    gathered = torch.gather(sentences[:, :T], 1, source_columns.clamp(min=0))
    inputs = torch.where(source_columns >= 0, gathered, torch.full_like(gathered, PAD_TOKEN))
    return inputs, targets, lengths

mixed_sentences = generate_sentences(N_TRAIN, mixed_probs, TRAIN_SEED)
X_train_tokens, y_train, mixed_lengths = make_last_token_task(mixed_sentences)
test_sentences = generate_sentences(N_TEST, mixed_probs, TEST_SEED)
X_test_tokens, y_test, test_lengths = make_last_token_task(test_sentences)
assert N_TEST % 4 == 0
assert torch.all((test_lengths >= 2 ** L) & (test_lengths <= D_MAX))
assert torch.all(y_test < V)

# Randomize equal-length ties without consulting targets or activations.
def split_length_quartiles(lengths, seed):
    lengths = np.asarray(lengths)
    if lengths.ndim != 1 or len(lengths) == 0 or len(lengths) % 4:
        raise ValueError('Expected a nonempty length vector divisible by four.')
    shuffled = np.random.default_rng(seed).permutation(len(lengths))
    order = shuffled[np.argsort(lengths[shuffled], kind='stable')]
    groups = dict(zip(QUARTILES, np.split(order, 4)))
    assert np.array_equal(np.sort(np.concatenate(list(groups.values()))), np.arange(len(lengths)))
    for left, right in zip(QUARTILES[:-1], QUARTILES[1:]):
        assert lengths[groups[left]].max() <= lengths[groups[right]].min()
    return groups

quartile_indices = split_length_quartiles(test_lengths.numpy(), QUARTILE_SEED)
quartile_summary = {}
for name, indices in quartile_indices.items():
    lengths = test_lengths[torch.from_numpy(indices)]
    quartile_summary[name] = dict(n=len(indices), min_length=int(lengths.min()),
                                  max_length=int(lengths.max()), mean_length=float(lengths.float().mean()))
    print(f'{name}: {quartile_summary[name]}')
assert torch.any(mixed_lengths < D_MAX) and torch.any(mixed_lengths == D_MAX)
assert torch.all(y_train < V)
print(f'device={DEVICE}; mixed train={N_TRAIN:,}; mixed test pool={N_TEST:,}')


## 2. Model loading, last-position activation extraction, and probing

In [ ]:
def infer_arch(state_dict):
    width = state_dict['token_embedding_table.weight'].shape[1]
    n_layers = max(int(k.split('.')[1]) for k in state_dict if k.startswith('blocks.')) + 1
    return dict(vocab_size=state_dict['token_embedding_table.weight'].shape[0],
                block_size=state_dict['blocks.0.attn.tril'].shape[0],
                embedding_dim=width, num_heads=NUM_HEADS,
                ffwd_size=state_dict['blocks.0.ffwd.net.0.weight'].shape[0] // width,
                num_layers=n_layers, dropout=0.0,
                rope='position_embedding_table.weight' not in state_dict, share_emb=False)

def load_model(path):
    saved = torch.load(path, map_location='cpu', weights_only=False)
    # Compiled checkpoints prefix parameter names with _orig_mod.
    state_dict = {key.removeprefix('_orig_mod.'): value
                  for key, value in saved['model'].items()}
    arch = infer_arch(state_dict); assert arch['block_size'] == T
    if (arch['num_layers'], arch['embedding_dim']) != (MODEL_DEPTH, 256):
        raise ValueError(f'Expected 12 layers and embedding width 256, got {arch}')
    model = models.CLM(**arch); model.load_state_dict(state_dict)
    return model.to(DEVICE).eval(), arch, int(saved.get('step', path.stem.rsplit('_t', 1)[-1]))

@torch.no_grad()
def extract_last_position_activations(model, inputs):
    # Write last-position vectors directly into one CPU array. On CPU a sliced
    # tensor otherwise retains the entire [batch, tokens, width] backing storage.
    from time import perf_counter
    started = perf_counter()
    width = model.token_embedding_table.weight.shape[1]
    activations = np.empty((model.num_layers + 1, len(inputs), width), dtype=np.float32)
    start, stop = 0, 0

    def save(depth, output):
        activations[depth, start:stop] = output[:, -1].detach().cpu().numpy()

    hooks = [model.blocks[0].register_forward_pre_hook(
        lambda module, args: save(0, args[0]))]
    for depth, block in enumerate(model.blocks, 1):
        hooks.append(block.register_forward_hook(
            lambda module, args, output, depth=depth: save(depth, output)))
    try:
        for start in range(0, len(inputs), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(inputs))
            model(inputs[start:stop].to(DEVICE))
            if start == 0 or stop % (BATCH_SIZE * 32) == 0 or stop == len(inputs):
                elapsed = perf_counter() - started
                print(f'  activations {stop:,}/{len(inputs):,}; {elapsed:.1f}s elapsed', flush=True)
    finally:
        for hook in hooks:
            hook.remove()
    return activations

def probe_across_depth(train_acts, test_acts):
    # One fitted probe per depth, shared by all four held-out populations.
    results = {'train_accuracy': [], 'train_cross_entropy_normalized': [],
               'tests': {name: {'test_accuracy': []} for name in QUARTILES}}
    train_labels = y_train.numpy()
    test_labels = y_test.numpy()
    for depth, X_train in enumerate(train_acts):
        probe = Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(
            C=PROBE_C, solver='lbfgs', max_iter=5000, random_state=0))])
        probe.fit(X_train, train_labels)
        results['train_accuracy'].append(probe.score(X_train, train_labels))
        results['train_cross_entropy_normalized'].append(
            log_loss(train_labels, probe.predict_proba(X_train), labels=np.arange(V)) / np.log(V))
        for name, indices in quartile_indices.items():
            accuracy = probe.score(test_acts[depth, indices], test_labels[indices])
            results['tests'][name]['test_accuracy'].append(accuracy)
            print(f'  depth {depth}, {name}: held-out accuracy={accuracy:.4f}', flush=True)
    counts = np.bincount(train_labels, minlength=V)
    results.update(chance_uniform=1/V, train_majority_baseline=counts.max()/counts.sum())
    return results

def valid_cache(payload):
    expected = dict(experiment=EXPERIMENT, probe_C=PROBE_C, n_train_mixed=N_TRAIN,
                    n_test_pool=N_TEST, train_seed=TRAIN_SEED, test_seed=TEST_SEED,
                    quartile_seed=QUARTILE_SEED, rules_seed=SEED_RULES,
                    quartile_summary=quartile_summary, probe_position='last input position')
    if any(payload.get(key) != value for key, value in expected.items()):
        return False
    for name in QUARTILES:
        values = np.asarray(payload.get('token_probe', {}).get('tests', {}).get(name, {}).get('test_accuracy', []))
        if values.shape != (MODEL_DEPTH + 1,) or not np.isfinite(values).all() or not ((values >= 0) & (values <= 1)).all():
            return False
    return True


## 3. Fit and evaluate stage 4

Fit the unchanged probe at each depth once, then evaluate on all four length quartiles. Extract test activations in original pool order and select quartile indices when scoring. `RECOMPUTE = False` reuses a compatible quartile cache only.


In [ ]:
token_probe_results = {}
architecture = None
for requested_step in STEPS:
    print(f'\n=== checkpoint t={requested_step} ===')
    if RESULT_PATHS[requested_step].exists() and not RECOMPUTE:
        with RESULT_PATHS[requested_step].open('rb') as handle: cached = pickle.load(handle)
        if valid_cache(cached) and cached.get('checkpoint_step') == requested_step:
            token_probe_results[requested_step] = cached['token_probe']
            print('reusing:', RESULT_PATHS[requested_step])
            continue
        print('incompatible cache; refitting this checkpoint')
    model, arch, loaded_step = load_model(CHECKPOINTS[requested_step])
    if loaded_step != requested_step: raise ValueError(f'Expected step {requested_step}, checkpoint reports {loaded_step}')
    if architecture is None: architecture = arch
    elif (arch['num_layers'], arch['embedding_dim']) != (architecture['num_layers'], architecture['embedding_dim']):
        raise ValueError('Checkpoint architectures differ.')
    train_acts = extract_last_position_activations(model, X_train_tokens)
    test_acts = extract_last_position_activations(model, X_test_tokens)
    result = probe_across_depth(train_acts, test_acts)
    token_probe_results[requested_step] = result
    payload = {'token_probe': result, 'checkpoint_step': requested_step, 'probe_C': PROBE_C,
               'experiment': EXPERIMENT, 'n_train_mixed': N_TRAIN, 'n_test_pool': N_TEST,
               'train_seed': TRAIN_SEED, 'test_seed': TEST_SEED, 'quartile_seed': QUARTILE_SEED,
               'quartile_summary': quartile_summary, 'quartile_indices': quartile_indices,
               'test_lengths': test_lengths.numpy(), 'rules_seed': SEED_RULES,
               'test_distribution': 'original mixed; equal-count empirical length quartiles',
               'probe_position': 'last input position'}
    with RESULT_PATHS[requested_step].open('wb') as handle: pickle.dump(payload, handle)
    print('saved:', RESULT_PATHS[requested_step])
    del model, train_acts, test_acts; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    elif DEVICE == 'mps': torch.mps.empty_cache()

## 4. Plot stage 4 held-out accuracy versus layer

This cell runs independently after a kernel restart. It loads the quartile result and exports a single accuracy figure as PDF and PNG, with four quartile curves plus a dashed black pure-ternary (length 27) reference, loaded from the existing stage 4 pure-ternary cache. The reference used the same training seed, sample count, probe settings, and checkpoint; the plotting cell checks those metadata. No pure-ternary refitting is needed. Quartile labels show the observed full-sequence length range. The embedding is shown as `emb`; the original Inside/Outside divider remains at L6.


In [ ]:
# Standalone plotting cell: no model loading or probe fitting.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd()
STAGE4_STEP, MODEL_DEPTH, V = 2097152, 12, 8
path = WORKSPACE / f'probe_results_step_{STAGE4_STEP}_mixed_train_length_quartiles_test_last_token_depth12.pkl'
if not path.is_file():
    raise FileNotFoundError(f'Run Sections 1–3 first; missing {path}')
with path.open('rb') as handle:
    payload = pickle.load(handle)
if (payload.get('experiment') != 'mixed_train_length_quartiles_test_depth12_v1'
        or payload.get('checkpoint_step') != STAGE4_STEP):
    raise ValueError(f'Incompatible result: {path}')

# Saved reference: ALL grammar levels ternary, so every full sequence has length 27.
# This is distinct from fixing only the root arity in the top-rule experiment.
ternary_path = WORKSPACE / f'probe_results_step_{STAGE4_STEP}_mixed_train_ternary_test_last_token_depth12.pkl'
if not ternary_path.is_file():
    raise FileNotFoundError(f'Missing saved pure-ternary reference: {ternary_path}')
with ternary_path.open('rb') as handle:
    ternary_payload = pickle.load(handle)
for key in ('checkpoint_step', 'probe_C', 'n_train_mixed', 'train_seed', 'probe_position'):
    if ternary_payload.get(key) != payload.get(key):
        raise ValueError(f'Pure-ternary reference has incompatible {key}: {ternary_path}')
ternary_accuracy = np.asarray(ternary_payload['token_probe']['test_accuracy'], dtype=float)
if (ternary_accuracy.shape != (MODEL_DEPTH + 1,) or not np.isfinite(ternary_accuracy).all()
        or not ((ternary_accuracy >= 0) & (ternary_accuracy <= 1)).all()):
    raise ValueError(f'Invalid pure-ternary accuracy: {ternary_path}')

plt.rcParams.update({'font.size': 9, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'pdf.fonttype': 42, 'ps.fonttype': 42})
depth = np.arange(MODEL_DEPTH + 1)
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, marker in zip(('Q1', 'Q2', 'Q3', 'Q4'), ('o', 's', '^', 'D')):
    values = np.asarray(payload['token_probe']['tests'][name]['test_accuracy'], dtype=float)
    if values.shape != (MODEL_DEPTH + 1,) or not np.isfinite(values).all() or not ((values >= 0) & (values <= 1)).all():
        raise ValueError(f'Invalid {name} accuracy: {path}')
    summary = payload['quartile_summary'][name]
    label = f"{name}: lengths {summary['min_length']}–{summary['max_length']}"
    ax.plot(depth, values, marker=marker, ms=4, lw=1.8, label=label)
ax.plot(depth, ternary_accuracy, color='black', ls='--', marker='x', ms=5, lw=2,
        label='Pure ternary: length 27')
ax.axhline(1/V, color='grey', ls=':', lw=1)
ax.axvline(6, color='black', ls='--', lw=1, alpha=.55)
ax.text(.43, .98, 'Inside', transform=ax.transAxes, ha='right', va='top')
ax.text(.57, .98, 'Outside', transform=ax.transAxes, ha='left', va='top')
ax.set(xticks=depth, xticklabels=['emb'] + [f'L{k}' for k in range(1, MODEL_DEPTH + 1)],
       xlabel='Transformer layer', ylabel='Held-out final-token probe accuracy',
       ylim=(0, 1.04), title='Stage 4: length quartiles and pure ternary')
ax.spines[['top', 'right']].set_visible(False)
fig.legend(*ax.get_legend_handles_labels(), loc='upper center', bbox_to_anchor=(.5, 1.06),
           ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, .82))
for ext in ('pdf', 'png'):
    output = WORKSPACE / f'token_probe_accuracy_stage4_length_quartiles_depth12.{ext}'
    fig.savefig(output, dpi=300, bbox_inches='tight')
    print('saved:', output)
plt.show()
